# pypgo.solver API Demo

This tutorial demonstrates the Python Newton solver API:
`pypgo.solver.solve_newton`, `SolveStatus`, `SolverResult`,
`SolveDiagnostics`, and `NewtonOptions`.

The Python API is intentionally small: users pass a `pypgo.energy`
potential energy and an initial state `x0`; the solver returns a new
solution array and diagnostics without mutating `x0`.


## Outline

1. Imports and a small quadratic problem
2. Basic Newton solve
3. Result fields and diagnostics
4. Input ownership: `x0` is not mutated
5. Fixed DOFs with implicit and explicit values
6. Line-search modes
7. `NewtonOptions` as a reusable parameter object
8. Solving a weighted `EnergySet`
9. Common validation errors
10. Public surface check


In [ ]:
import numpy as np
import pypgo as pgo
import pypgo.solver as solver


## 1. A small quadratic problem

We solve:

`min_x 1/2 x^T A x + b^T x`

with `A = I` and `b = [-1, 2, -4]`.  The exact minimizer is
`x* = -b = [1, -2, 4]`.


In [ ]:
A = np.eye(3, dtype=np.float64)
b = np.array([-1.0, 2.0, -4.0], dtype=np.float64)
energy = pgo.energy.QuadraticEnergy(A, b=b)

x0 = np.array([10.0, -3.0, 5.0], dtype=np.float64)
exact = -b

print("energy:", energy)
print("x0:", x0)
print("exact minimizer:", exact)


## 2. Basic Newton solve

`solve_newton` returns a `SolverResult` object.  The default line search
is `"backtrack"` and the default tolerance is `1e-6`.


In [ ]:
result = solver.solve_newton(
    energy,
    x0=x0,
    max_iter=50,
    tol=1e-8,
    damping=False,
    line_search="backtrack",
)

print("status:", result.status)
print("converged:", result.converged)
print("iterations:", result.iterations)
print("x:", result.x)
print("close to exact:", np.allclose(result.x, exact))


## 3. Result fields and diagnostics

`SolverResult` separates solver-level status from optimization-level
outputs such as the final objective and final solution vector.


In [ ]:
print("raw_status_code:", result.raw_status_code)
print("final_objective:", result.final_objective)
print("final_gradient_norm:", result.final_gradient_norm)
print("final_gradient_max_norm:", result.final_gradient_max_norm)

diag = result.diagnostics
print("min_feasible_alpha:", diag.min_feasible_alpha)
print("min_line_search_alpha:", diag.min_line_search_alpha)
print("min_effective_alpha:", diag.min_effective_alpha)
print("material clamps:", diag.material_clamp_count)
print("contact clamps:", diag.contact_clamp_count)


## 4. Input ownership

The solver treats `x0` as a read-only initial state.  It copies `x0`
internally and returns an independently owned `result.x`.


In [ ]:
before = x0.copy()
result = solver.solve_newton(energy, x0=x0, damping=False)

print("x0 unchanged:", np.array_equal(x0, before))
print("result.x shares memory with x0:", np.shares_memory(result.x, x0))

result.x[0] = 123.0
again = solver.solve_newton(energy, x0=x0, damping=False)
print("mutating one result does not affect a fresh solve:", again.x)


## 5. Fixed DOFs

`fixed_dofs` pins selected variables.  If `fixed_values=None`, each fixed
value is taken from `x0[fixed_dofs]`.  DOFs may be unsorted; the service
canonicalizes them before calling the native Newton service.


In [ ]:
x0_fixed = np.array([7.0, 10.0, -3.0], dtype=np.float64)

implicit = solver.solve_newton(
    energy,
    x0=x0_fixed,
    fixed_dofs=[2, 0],
    fixed_values=None,
    damping=False,
)

print("implicit fixed values:", implicit.x)
print("x[0] fixed to x0[0]:", implicit.x[0])
print("x[2] fixed to x0[2]:", implicit.x[2])


In [ ]:
explicit = solver.solve_newton(
    energy,
    x0=x0,
    fixed_dofs=[2],
    fixed_values=np.array([9.0], dtype=np.float64),
    damping=False,
)

print("explicit fixed value:", explicit.x)


## 6. Line-search modes

Python exposes four stable keywords:
`"golden"`, `"brents"`, `"backtrack"`, and `"simple"`.


In [ ]:
for line_search in ("golden", "brents", "backtrack", "simple"):
    r = solver.solve_newton(
        energy,
        x0=x0,
        line_search=line_search,
        damping=False,
    )
    print(f"{line_search:10s}", r.status.name, r.iterations, r.x)


## 7. `NewtonOptions`

`NewtonOptions` is a small frozen dataclass.  It is useful for keeping
solver parameters near an experiment, and it can be passed directly to
`solve_newton`.


In [ ]:
opts = solver.NewtonOptions(
    max_iter=20,
    tol=1e-8,
    damping=False,
    line_search="backtrack",
    verbose=0,
)

result = solver.solve_newton(
    energy,
    x0=x0,
    options=opts,
)
print(result)


## 8. Solving a weighted EnergySet

`solve_newton` accepts any object with a `pypgo.energy.PotentialEnergy`
handle, including `EnergySet`.  This example combines two quadratic
terms with different weights.


In [ ]:
attraction = pgo.energy.QuadraticEnergy(
    np.eye(3, dtype=np.float64),
    b=np.array([-2.0, 0.0, 0.0], dtype=np.float64),
)
regularizer = pgo.energy.QuadraticEnergy(0.1 * np.eye(3, dtype=np.float64))

total = pgo.energy.EnergySet([
    (attraction, 1.0),
    (regularizer, 1.0),
])

r = solver.solve_newton(total, x0=total.zero_state(), damping=False)
print("EnergySet solution:", r.x)
print("final objective:", r.final_objective)


## 9. Common validation errors

Invalid line-search names and inconsistent fixed values raise
`ValueError`.


In [ ]:
try:
    solver.solve_newton(energy, x0=x0, line_search="wolfe")
except ValueError as exc:
    print("invalid line_search:", exc)

try:
    solver.solve_newton(
        energy,
        x0=x0,
        fixed_dofs=[0, 1],
        fixed_values=np.array([1.0], dtype=np.float64),
    )
except ValueError as exc:
    print("fixed_values mismatch:", exc)


## 10. Public surface

The first Python release intentionally hides legacy C++ names such as
`NewtonSolver`, `SolverParam`, and `EnergyOptimizer`.


In [ ]:
public_names = [name for name in dir(solver) if not name.startswith("_")]
print(public_names)

hidden = {"NewtonSolver", "SolverParam", "EnergyOptimizer", "minimize"}
print("hidden legacy names present:", sorted(hidden.intersection(public_names)))
